# Assignment 5: RNN vs LSTM vs GRU for sequence classification
Binary sentiment classification on the IMDB reviews dataset, compared with accuracy, precision, recall, F1, ROC-AUC and training time.

In [ ]:
import time, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
tf.keras.utils.set_random_seed(42)

VOCAB, MAXLEN = 10000, 200
(X_tr, y_tr), (X_te, y_te) = keras.datasets.imdb.load_data(num_words=VOCAB)
X_tr = keras.utils.pad_sequences(X_tr, maxlen=MAXLEN); X_te = keras.utils.pad_sequences(X_te, maxlen=MAXLEN)
print(X_tr.shape, X_te.shape, "positive ratio:", y_tr.mean())

In [ ]:
def build_model(kind):
    layer = {"RNN": keras.layers.SimpleRNN, "LSTM": keras.layers.LSTM, "GRU": keras.layers.GRU}[kind]
    m = keras.Sequential([
        keras.layers.Input((MAXLEN,)),
        keras.layers.Embedding(VOCAB, 64),
        layer(64),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
    return m

results, histories, preds = [], {}, {}
for kind in ["RNN", "LSTM", "GRU"]:
    print(f"\n=== {kind} ===")
    m = build_model(kind)
    t0 = time.time()
    h = m.fit(X_tr, y_tr, validation_split=0.2, epochs=5, batch_size=128, verbose=1)
    train_time = time.time() - t0
    prob = m.predict(X_te, verbose=0).ravel(); pred = (prob > 0.5).astype(int)
    histories[kind], preds[kind] = h.history, pred
    results.append({"Model": kind, "Params": m.count_params(),
                    "Accuracy": accuracy_score(y_te, pred), "Precision": precision_score(y_te, pred),
                    "Recall": recall_score(y_te, pred), "F1": f1_score(y_te, pred),
                    "ROC-AUC": roc_auc_score(y_te, prob), "Train time (s)": train_time})
res = pd.DataFrame(results).set_index("Model").round(4)
res

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for k, h in histories.items():
    ax[0].plot(h["val_loss"], label=k); ax[1].plot(h["val_accuracy"], label=k)
ax[0].set_title("Validation loss"); ax[1].set_title("Validation accuracy")
for a in ax: a.set_xlabel("epoch"); a.legend()
plt.show()

fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
for a, k in zip(ax, preds):
    sns.heatmap(confusion_matrix(y_te, preds[k]), annot=True, fmt="d", cmap="Blues", ax=a, cbar=False)
    a.set_title(f"{k} confusion matrix"); a.set_xlabel("Predicted"); a.set_ylabel("Actual")
plt.tight_layout(); plt.show()

res[["Accuracy", "F1", "ROC-AUC"]].plot(kind="bar", figsize=(7, 4), ylim=(0.5, 1)); plt.title("Metric comparison"); plt.show()

## Analysis
- **SimpleRNN** suffers from vanishing gradients, so it struggles with long reviews (200 tokens) and usually scores lowest and is unstable.
- **LSTM** uses gates and a cell state to retain long-range information: typically the best accuracy but slowest and most parameters.
- **GRU** has fewer gates than LSTM: accuracy close to LSTM with faster training.
(Exact numbers vary run to run; read them from the table above.)